# Notebook 04 — Deep Learning Pipeline
## ResNet50 Features → Cosine ReID → DeepSORT-Style Tracker

### Pipeline Overview
```
Detection crop (64×128 RGB)
        ↓
   ResNet50 backbone (pretrained, ImageNet)
   Global Average Pooling → 2048-d vector
   L2 normalise → unit-sphere embedding
        ↓
   Cosine distance matrix between
   track templates and new detections
        ↓
   Combined score = w_iou * IoU + w_app * (1 - cos_dist)
        ↓
   Hungarian assignment
        ↓
   EMA template update + Kalman motion
```

### Mathematical Foundations

#### 1. ResNet50 Feature Extraction
ResNet50 with residual connections:
$$\mathbf{H}(x) = \mathcal{F}(x, \{W_i\}) + x$$
where $\mathcal{F}$ is the residual mapping.
After Global Average Pooling: $\mathbf{f} = \text{GAP}(\text{ResNet50}(\text{img})) \in \mathbb{R}^{2048}$

#### 2. L2 Normalisation (Unit Sphere Embedding)
$$\hat{\mathbf{f}} = \frac{\mathbf{f}}{\|\mathbf{f}\|_2 + \epsilon}$$
This maps all features to the unit sphere,
so **cosine similarity = dot product**.

#### 3. Cosine Distance
$$d_{\cos}(\hat{\mathbf{f}}_1, \hat{\mathbf{f}}_2) = 1 - \hat{\mathbf{f}}_1 \cdot \hat{\mathbf{f}}_2$$
Range: $[0, 2]$ where 0 = identical, 2 = opposite.
After L2 norm: $d_{\cos} \in [0, 2]$, we convert to similarity $\in [0, 1]$:
$$\text{sim} = \frac{2 - d_{\cos}}{2} = \frac{1 + \hat{\mathbf{f}}_1 \cdot \hat{\mathbf{f}}_2}{2}$$

#### 4. Two-Stage Matching (DeepSORT-Style)
Stage 1 — High-confidence detections (score ≥ 0.6):
$$S = w_{\text{IoU}} \cdot \text{IoU} + w_{\text{app}} \cdot \text{cosine sim}$$

Stage 2 — Remaining unmatched tracks + low-confidence detections:
$$S = \text{IoU only}$$

#### 5. EMA Template Update
Same as classical pipeline:
$$T_t = \alpha \cdot T_{t-1} + (1-\alpha) \cdot \hat{\mathbf{f}}_t, \quad \alpha=0.9$$
After update, re-normalise: $T_t \leftarrow T_t / \|T_t\|_2$

In [2]:
import os, json, cv2, random
import numpy as np
import pandas as pd
from tqdm import tqdm
from collections import defaultdict
from scipy.optimize import linear_sum_assignment
import torch
import torch.nn as nn
import torch.nn.functional as F
import torchvision.models as models
import torchvision.transforms as T
from PIL import Image
import matplotlib.pyplot as plt
import warnings; warnings.filterwarnings('ignore')

with open(os.path.join(
    r'D:\MTech\Sem-2\IT585-Advanced_ML\AML-project\VisDrone',
    'VisDrone_outputs', 'config.json')) as f:
    cfg = json.load(f)

TRAIN_DIR     = cfg['TRAIN_DIR']
VAL_DIR       = cfg['VAL_DIR']
OUTPUT_DIR    = cfg['OUTPUT_DIR']
VALID_CLASSES = set(cfg['VALID_CLASSES'])
VAL_SEQS      = cfg['VAL_SEQS']
DEVICE        = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Device: {DEVICE}')
if DEVICE.type == 'cuda':
    print(f'GPU: {torch.cuda.get_device_name(0)}')

Device: cuda
GPU: NVIDIA GeForce RTX 4050 Laptop GPU


## Part 1 — ResNet50 Feature Extractor

In [3]:
# ──────────────────────────────────────────────────────────────────
# DEEP FEATURE EXTRACTOR
# Uses ResNet50 pretrained on ImageNet as backbone.
# Removes the final classification layer.
# Adds Global Average Pooling → 2048-d feature.
# L2-normalises to unit sphere for cosine similarity matching.
# ──────────────────────────────────────────────────────────────────
class DeepFeatureExtractor(nn.Module):
    """
    ResNet50 feature extractor for person Re-ID.

    Architecture:
        ResNet50 (pretrained ImageNet)
        → Remove final FC layer
        → Global Average Pooling: (B, 2048, H, W) → (B, 2048)
        → L2 normalise: ||f||_2 = 1

    Why ResNet50?
        - Strong ImageNet pretraining = good visual features out-of-box
        - 2048-d output is richer than classical 2058-d but encodes
          semantic appearance (not just colour/texture histograms)
        - ResNet residual connections prevent vanishing gradients
    """
    def __init__(self, device):
        super().__init__()
        # Load pretrained ResNet50
        backbone = models.resnet50(weights=models.ResNet50_Weights.IMAGENET1K_V1)
        # Remove final FC layer (classifier)
        self.backbone = nn.Sequential(*list(backbone.children())[:-2])
        # Global Average Pooling
        self.gap = nn.AdaptiveAvgPool2d(1)
        self.device = device
        self.to(device)
        self.eval()  # Always in eval mode for feature extraction

    @torch.no_grad()
    def forward(self, x):
        """
        x: (B, 3, 256, 128) normalised image tensor
        returns: (B, 2048) L2-normalised feature vector
        """
        feat = self.backbone(x)       # (B, 2048, H', W')
        feat = self.gap(feat)          # (B, 2048, 1, 1)
        feat = feat.view(x.size(0), -1)  # (B, 2048)
        # L2 normalise: f_hat = f / ||f||_2
        feat = F.normalize(feat, p=2, dim=1)
        return feat

    def extract_batch(self, bgr_crops):
        """
        Extract features from a list of BGR numpy crops.

        Args:
            bgr_crops: list of numpy arrays (H, W, 3) BGR

        Returns:
            numpy array (N, 2048)
        """
        if len(bgr_crops) == 0:
            return np.empty((0, 2048), dtype=np.float32)

        # Preprocessing: BGR → RGB → resize → normalise
        transform = T.Compose([
            T.Resize((256, 128)),
            T.ToTensor(),
            T.Normalize(mean=[0.485, 0.456, 0.406],
                        std=[0.229, 0.224, 0.225])
        ])

        tensors = []
        for crop in bgr_crops:
            rgb = cv2.cvtColor(crop, cv2.COLOR_BGR2RGB)
            pil = Image.fromarray(rgb)
            tensors.append(transform(pil))

        batch = torch.stack(tensors).to(self.device)
        feats = self.forward(batch)
        return feats.cpu().numpy()


# Instantiate the feature extractor
deep_extractor = DeepFeatureExtractor(DEVICE)

# Quick test
dummy_bgr = [np.random.randint(0, 255, (128, 64, 3), dtype=np.uint8)]
test_feat = deep_extractor.extract_batch(dummy_bgr)
print(f'Deep feature shape: {test_feat.shape}')
print(f'L2 norm (should be 1.0): {np.linalg.norm(test_feat[0]):.6f}')

# Count model parameters
params = sum(p.numel() for p in deep_extractor.parameters())
print(f'ResNet50 parameters: {params:,}')

Deep feature shape: (1, 2048)
L2 norm (should be 1.0): 1.000000
ResNet50 parameters: 23,508,032


## Part 2 — DeepSORT-Style Tracker

In [4]:
# ──────────────────────────────────────────────────────────────────
# KALMAN FILTER — same as classical pipeline
# (Re-defined here so this notebook is self-contained)
# ──────────────────────────────────────────────────────────────────
class KalmanBoxTracker:
    count = 0
    def __init__(self, bbox, feature=None):
        self.F = np.eye(7); self.F[0,4]=self.F[1,5]=self.F[2,6]=1
        self.H = np.zeros((4,7)); self.H[:4,:4]=np.eye(4)
        self.Q = np.eye(7); self.Q[4:,4:]*=0.01
        self.R = np.eye(4); self.R[2:,2:]*=10
        self.P = np.eye(7)*10; self.P[4:,4:]*=1000
        x1,y1,x2,y2=bbox; cx=(x1+x2)/2; cy=(y1+y2)/2
        w=x2-x1; h=y2-y1
        self.x = np.array([[cx],[cy],[w*h],[w/(h+1e-8)],[0],[0],[0]])
        KalmanBoxTracker.count += 1
        self.id=KalmanBoxTracker.count; self.age=0
        self.hits=0; self.hit_streak=0; self.time_since_update=0
        self.template = feature.copy() if feature is not None else None

    def predict(self):
        if self.x[2]+self.x[6]<=0: self.x[6]=0
        self.x=self.F@self.x; self.P=self.F@self.P@self.F.T+self.Q
        self.age+=1; self.time_since_update+=1
        return self.get_bbox()

    def update(self, bbox, feature=None, ema_alpha=0.9):
        x1,y1,x2,y2=bbox; cx=(x1+x2)/2; cy=(y1+y2)/2
        w=x2-x1; h=y2-y1
        z=np.array([[cx],[cy],[w*h],[w/(h+1e-8)]])
        S=self.H@self.P@self.H.T+self.R; K=self.P@self.H.T@np.linalg.inv(S)
        self.x=self.x+K@(z-self.H@self.x)
        self.P=(np.eye(7)-K@self.H)@self.P
        self.time_since_update=0; self.hits+=1; self.hit_streak+=1
        if feature is not None:
            if self.template is not None:
                # EMA update: T_t = α*T_{t-1} + (1-α)*f_t
                self.template = ema_alpha*self.template+(1-ema_alpha)*feature
                # Re-normalise after EMA to keep on unit sphere
                norm = np.linalg.norm(self.template)
                if norm > 0: self.template /= norm
            else:
                self.template = feature.copy()

    def get_bbox(self):
        cx,cy,s,r=self.x[0,0],self.x[1,0],self.x[2,0],self.x[3,0]
        w=np.sqrt(max(s*r,0)); h=s/(w+1e-8)
        return [cx-w/2,cy-h/2,cx+w/2,cy+h/2]


# ──────────────────────────────────────────────────────────────────
# DEEPSORT-STYLE TRACKER
# Two-stage matching:
#   Stage 1: High-conf dets + all tracks using IoU + cosine sim
#   Stage 2: Remaining unmatched using IoU only
#
# Cosine similarity (L2-normalised features):
#   sim(f1, f2) = f1 · f2  (dot product, since ||f|| = 1)
#   range: [-1, 1] → clipped to [0, 1]
# ──────────────────────────────────────────────────────────────────
def compute_iou_matrix(bboxes_a, bboxes_b):
    N,M=len(bboxes_a),len(bboxes_b); iou=np.zeros((N,M))
    for i,a in enumerate(bboxes_a):
        for j,b in enumerate(bboxes_b):
            xi1=max(a[0],b[0]); yi1=max(a[1],b[1])
            xi2=min(a[2],b[2]); yi2=min(a[3],b[3])
            inter=max(0,xi2-xi1)*max(0,yi2-yi1)
            area_a=(a[2]-a[0])*(a[3]-a[1])
            area_b=(b[2]-b[0])*(b[3]-b[1])
            iou[i,j]=inter/(area_a+area_b-inter+1e-8)
    return iou


class DeepSORT:
    """
    DeepSORT-style tracker using ResNet50 cosine similarity.

    Key difference from ClassicalSORT:
      - Uses L2-normalised deep features (ResNet50 2048-d)
      - Cosine similarity instead of XGBoost probability
      - Two-stage matching: appearance first, IoU fallback
      - Re-normalises EMA template after each update

    Combined score per stage:
      Stage 1: S = w_iou * IoU + w_app * clip(f_track · f_det, 0, 1)
      Stage 2: S = IoU  (for tracks that missed stage 1)
    """

    def __init__(self, max_age=20, min_hits=3, iou_threshold=0.3,
                  w_iou=0.5, w_app=0.5, ema_alpha=0.9,
                  high_conf_threshold=0.6):
        self.max_age    = max_age
        self.min_hits   = min_hits
        self.iou_thr    = iou_threshold
        self.w_iou      = w_iou
        self.w_app      = w_app
        self.ema_alpha  = ema_alpha
        self.high_conf  = high_conf_threshold
        self.trackers   = []
        self.frame_count = 0
        KalmanBoxTracker.count = 0

    def _cosine_sim_matrix(self, track_templates, det_features):
        """
        Compute cosine similarity matrix between track templates
        and detection features.

        Since both are L2-normalised:
            sim(T, D) = T @ D^T  (matrix dot product)

        Returns:
            (N_tracks, N_dets) matrix, values in [0, 1] (clipped)
        """
        T = np.array(track_templates, dtype=np.float32)  # (N, 2048)
        D = np.array(det_features,    dtype=np.float32)  # (M, 2048)
        sim = T @ D.T  # (N, M) — dot product = cosine sim for unit vectors
        return np.clip(sim, 0, 1)  # clip to [0, 1]

    def update(self, detections, features):
        """
        Process one frame with two-stage matching.

        Args:
            detections: (N, 4) array [x1, y1, x2, y2]
            features  : (N, 2048) L2-normalised deep features

        Returns:
            List of [x1, y1, x2, y2, track_id]
        """
        self.frame_count += 1
        predicted_bboxes = [t.predict() for t in self.trackers]

        if len(detections) == 0:
            self.trackers = [t for t in self.trackers
                             if t.time_since_update <= self.max_age]
            return []

        if len(self.trackers) == 0:
            for i, det in enumerate(detections):
                feat = features[i] if features is not None else None
                self.trackers.append(KalmanBoxTracker(det, feat))
            return []

        # ── Stage 1: Full matching with appearance ────────────────
        iou_mat = compute_iou_matrix(predicted_bboxes, detections)

        # Cosine similarity matrix
        templates = [t.template for t in self.trackers]
        has_template = [t is not None for t in templates]

        app_mat = np.zeros_like(iou_mat)
        if features is not None and any(has_template):
            valid_tracks = [i for i, h in enumerate(has_template) if h]
            valid_templates = [templates[i] for i in valid_tracks]
            cos_sim = self._cosine_sim_matrix(valid_templates,
                                               features.tolist())
            for row, track_idx in enumerate(valid_tracks):
                app_mat[track_idx] = cos_sim[row]

        score_mat = self.w_iou * iou_mat + self.w_app * app_mat
        row_ind, col_ind = linear_sum_assignment(-score_mat)

        matched = []
        unmatched_dets = list(range(len(detections)))
        unmatched_tracks = list(range(len(self.trackers)))

        for r, c in zip(row_ind, col_ind):
            if score_mat[r, c] >= self.iou_thr:
                matched.append((r, c))
                if c in unmatched_dets: unmatched_dets.remove(c)
                if r in unmatched_tracks: unmatched_tracks.remove(r)

        # ── Stage 2: IoU-only for remaining unmatched ─────────────
        if unmatched_tracks and unmatched_dets:
            sub_iou = iou_mat[
                np.ix_(unmatched_tracks, unmatched_dets)]
            r2, c2 = linear_sum_assignment(-sub_iou)
            for r, c in zip(r2, c2):
                real_r = unmatched_tracks[r]
                real_c = unmatched_dets[c]
                if iou_mat[real_r, real_c] >= self.iou_thr:
                    matched.append((real_r, real_c))
                    unmatched_dets.remove(real_c)

        # Update matched trackers
        matched_track_ids = set()
        for r, c in matched:
            feat = features[c] if features is not None else None
            self.trackers[r].update(detections[c], feat, self.ema_alpha)
            matched_track_ids.add(r)

        # Create new trackers
        for c in unmatched_dets:
            feat = features[c] if features is not None else None
            self.trackers.append(KalmanBoxTracker(detections[c], feat))

        # Mark unmatched
        for i in range(len(predicted_bboxes)):
            if i not in matched_track_ids:
                self.trackers[i].hit_streak = 0

        self.trackers = [t for t in self.trackers
                         if t.time_since_update <= self.max_age]

        results = []
        for t in self.trackers:
            if (t.time_since_update == 0 and
                    (t.hit_streak >= self.min_hits or
                     self.frame_count <= self.min_hits)):
                results.append(t.get_bbox() + [t.id])
        return results


print('DeepSORT tracker defined!')

DeepSORT tracker defined!


In [5]:
# ──────────────────────────────────────────────────────────────────
# RUN DEEP TRACKER ON A SEQUENCE
# Same interface as the classical tracker for fair comparison.
# ──────────────────────────────────────────────────────────────────
def read_visdrone_annotation(anno_path, valid_classes=None):
    cols = ['frame_id','target_id','x','y','w','h',
            'score','class_id','truncation','occlusion']
    df = pd.read_csv(anno_path, header=None, names=cols)
    df = df[df['score']==1]
    df = df[df['target_id']>0]
    if valid_classes: df=df[df['class_id'].isin(valid_classes)]
    return df[(df['w']>0)&(df['h']>0)].reset_index(drop=True)


def run_deep_tracker_on_sequence(split_dir, seq_name, extractor,
                                   valid_classes, dropped_frames=None,
                                   save_dir=None):
    """
    Run the DeepSORT tracker on one sequence.

    Args:
        split_dir     : TRAIN_DIR or VAL_DIR
        seq_name      : sequence name
        extractor     : DeepFeatureExtractor instance
        valid_classes : set of class IDs to keep
        dropped_frames: set of frame IDs to skip
        save_dir      : directory to save results

    Returns:
        list of [frame_id, track_id, x1, y1, x2, y2]
    """
    anno_path = os.path.join(split_dir, 'annotations', seq_name + '.txt')
    seq_dir   = os.path.join(split_dir, 'sequences', seq_name)
    df = read_visdrone_annotation(anno_path, valid_classes)

    if dropped_frames is None:
        dropped_frames = set()

    tracker = DeepSORT()
    track_results = []
    all_frame_ids = sorted(df['frame_id'].unique())

    for fid in tqdm(all_frame_ids, desc=f'{seq_name}', leave=False):
        if fid in dropped_frames:
            tracker.update(np.empty((0,4)), None)
            continue

        img_path = os.path.join(seq_dir, f'{fid:07d}.jpg')
        if not os.path.exists(img_path):
            tracker.update(np.empty((0,4)), None)
            continue

        img = cv2.imread(img_path)
        if img is None:
            tracker.update(np.empty((0,4)), None)
            continue
        H_img, W_img = img.shape[:2]

        frame_df = df[df['frame_id']==fid]
        detections, crops = [], []
        for _, row in frame_df.iterrows():
            x1=max(0,int(row['x'])); y1=max(0,int(row['y']))
            x2=min(W_img,int(row['x']+row['w']))
            y2=min(H_img,int(row['y']+row['h']))
            if (x2-x1)*(y2-y1)<400: continue
            detections.append([x1,y1,x2,y2])
            crops.append(cv2.resize(img[y1:y2,x1:x2],(64,128)))

        if not detections:
            tracker.update(np.empty((0,4)), None)
            continue

        detections = np.array(detections, dtype=np.float32)
        features   = extractor.extract_batch(crops)  # (N, 2048)

        tracks = tracker.update(detections, features)
        for t in tracks:
            x1,y1,x2,y2,tid = t
            track_results.append([fid,int(tid),x1,y1,x2,y2])

    if save_dir:
        os.makedirs(save_dir, exist_ok=True)
        drop_label = f'drop{len(dropped_frames)}' if dropped_frames else 'nodrop'
        out_path = os.path.join(save_dir, f'{seq_name}_deep_{drop_label}.txt')
        with open(out_path, 'w') as f:
            for row in track_results:
                f.write(','.join(map(str,row))+'\n')

    return track_results


# Run on first validation sequence
test_seq = VAL_SEQS[0]
save_dir = os.path.join(OUTPUT_DIR, 'tracks', 'deep')
print(f'Running DeepSORT on: {test_seq}')
results = run_deep_tracker_on_sequence(
    VAL_DIR, test_seq, deep_extractor, VALID_CLASSES, save_dir=save_dir
)
print(f'\nTotal track detections : {len(results):,}')
print(f'Unique track IDs       : {len(set(r[1] for r in results))}')
print('\n✅ Deep Learning pipeline complete!')
print('Next: Notebook 05 — Missing Frame Experiment')

Running DeepSORT on: uav0000086_00000_v



Total track detections : 21,217
Unique track IDs       : 67

✅ Deep Learning pipeline complete!
Next: Notebook 05 — Missing Frame Experiment
